# 05 — A/B testování prompt variant

Změnili jste prompt — zlepšili jste ho? Jediná správná odpověď: **měřil jsem to**.

Tenhle notebook pustí stejný offline eval (golden dataset z notebooku `03`) proti více variantám systémového promptu a vytvoří v LangSmith oddělené experimenty, které pak v UI porovnáš side-by-side.

Předpoklady:
- Dataset `kosik-eval-golden` už existuje (jinak: `python scripts/seed_eval_dataset.py`).
- `LANGSMITH_API_KEY` a `OPENAI_API_KEY` v `.env`.

Trvá ~3–5 minut pro 3 varianty × 12 příkladů × 4 evaluators.

In [1]:
from kosik_workshop.config import load_env
load_env()

## 1. Prompt varianty

Soubor `src/kosik_workshop/prompts/variants.py` drží 3 cílené úpravy oproti baseline:

| Variant | Hypotéza |
|---|---|
| `v1_baseline` | Kopie current production promptu — kontrolní vzorek. |
| `v2_strict_citations` | Přísnější citation rule → měl by zlepšit `cites_product_id`. |
| `v3_loud_allergens` | Hlasitější allergen warning → měl by zlepšit `allergen_flagged_explicitly`. |

Pravidlo prompt engineeringu: **měň jednu věc**. Když měníš dvě, nevíš, která dělá rozdíl.

In [2]:
from kosik_workshop.prompts.variants import PROMPT_VARIANTS

for name, text in PROMPT_VARIANTS.items():
    print(f'=== {name} ({len(text)} chars) ===')
    print(text[:200], '...\n')

=== v1_baseline (1556 chars) ===
Jsi nákupní asistent e-shopu Košík.cz. Pomáháš zákazníkům najít produkty, porovnat varianty a zvládnout nákup s ohledem na jejich preference a alergeny.

## Co umíš
- Vyhledávat produkty v katalogu po ...

=== v2_strict_citations (1691 chars) ===
Jsi nákupní asistent e-shopu Košík.cz. Pomáháš zákazníkům najít produkty, porovnat varianty a zvládnout nákup s ohledem na jejich preference a alergeny.

## Co umíš
- Vyhledávat produkty v katalogu po ...

=== v3_loud_allergens (1727 chars) ===
Jsi nákupní asistent e-shopu Košík.cz. Pomáháš zákazníkům najít produkty, porovnat varianty a zvládnout nákup s ohledem na jejich preference a alergeny.

## Co umíš
- Vyhledávat produkty v katalogu po ...



In [3]:
# Diff baseline vs v2 — uvidíš, jaký řádek se reálně změnil.
import difflib

diff = difflib.unified_diff(
    PROMPT_VARIANTS['v1_baseline'].splitlines(keepends=True),
    PROMPT_VARIANTS['v2_strict_citations'].splitlines(keepends=True),
    fromfile='v1_baseline',
    tofile='v2_strict_citations',
    n=1,
)
print(''.join(diff))

--- v1_baseline
+++ v2_strict_citations
@@ -13,3 +13,3 @@
 - Vykáš, jsi věcný a stručný.
-- Ceny uváděj v Kč, u doporučení vždy zmiň `product_id` (slug), aby se dal produkt jednoznačně dohledat.
+- Ceny uváděj v Kč. U **každé** zmínky o konkrétním produktu MUSÍŠ uvést `product_id` (slug) v závorce hned za názvem. Příklad: *Madeta Jihočeské máslo 250 g (madeta-jihoceske-maslo-250-g) — 49 Kč*. Bez `product_id` odpověď není kompletní.
 - Používej krátký markdown — odrážky u seznamů produktů, tučně u klíčového údaje.



## 2. Spuštění A/B evaluace

Pro každou variantu:
1. Postavíme čerstvý agent s daným system promptem.
2. Pustíme `run_evaluation()` proti `kosik-eval-golden`.
3. Zapíšeme společný `ab_group` label do experiment metadata — anchor pro pozdější UI compare.

Skript `scripts/run_ab_eval.py` dělá to samé z příkazové řádky.

In [4]:
from datetime import datetime, timezone
from kosik_workshop.agent import build_agent
from kosik_workshop.evals.runner import run_evaluation
from kosik_workshop.prompts.variants import get_variant

ab_group = f'nb-{datetime.now(timezone.utc):%H%M}'
variants_to_test = ['v1_baseline', 'v2_strict_citations', 'v3_loud_allergens']

experiments = {}
for variant in variants_to_test:
    print(f'\n--- {variant} ---')
    agent = build_agent(system_text=get_variant(variant))
    result = run_evaluation(
        agent,
        experiment_prefix=f'{variant}__{ab_group}',
        metadata={
            'ab_group': ab_group,
            'prompt_variant': variant,
        },
        description=f"A/B test '{ab_group}' — variant {variant}",
    )
    experiments[variant] = result

print(f'\n✅ A/B group: {ab_group}')


--- v1_baseline ---


/Users/petrmalik/projects/kosik-workshop/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'v1_baseline__nb-1750-1c960ab4' at:
https://eu.smith.langchain.com/o/917e9029-f367-49b9-87b0-fe366fffe6f5/datasets/cc9b35b1-88e6-4911-8f6b-b680f7595237/compare?selectedSessions=04281b94-0e6b-4ee0-946f-c9a69c23b1e6




13it [01:03,  4.91s/it]



--- v2_strict_citations ---
View the evaluation results for experiment: 'v2_strict_citations__nb-1750-a88fc0ae' at:
https://eu.smith.langchain.com/o/917e9029-f367-49b9-87b0-fe366fffe6f5/datasets/cc9b35b1-88e6-4911-8f6b-b680f7595237/compare?selectedSessions=ac72fa28-9914-4f71-834e-866c96cd44dd




13it [00:57,  4.40s/it]



--- v3_loud_allergens ---
View the evaluation results for experiment: 'v3_loud_allergens__nb-1750-86da9000' at:
https://eu.smith.langchain.com/o/917e9029-f367-49b9-87b0-fe366fffe6f5/datasets/cc9b35b1-88e6-4911-8f6b-b680f7595237/compare?selectedSessions=2c89ea60-1d6b-4873-aff9-fcf3fb8bf518




13it [00:59,  4.56s/it]


✅ A/B group: nb-1750


## 3. Lokální porovnání skóre

Sesumujeme per-evaluator skóre napříč variantami do jedné tabulky.
Stejný pohled máš v LangSmith UI (Compare view), tady je to pro rychlou kontrolu.

In [5]:
import pandas as pd

summary_rows = []
for variant, result in experiments.items():
    df = result.to_pandas()
    score_cols = [c for c in df.columns if c.startswith('feedback.')]
    means = df[score_cols].mean()
    row = {'variant': variant}
    for col in score_cols:
        row[col.replace('feedback.', '')] = round(float(means[col]), 3)
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index('variant')
summary

,tool_called_correctly,cites_product_id,allergen_flagged_explicitly,no_hallucinated_products
variant,,,,
v1_baseline,0.917,0.833,1.000,1.000
v2_strict_citations,0.917,1.000,1.000,1.000
v3_loud_allergens,0.833,0.917,0.917,0.917


In [ ]:
# Delta proti baseline — kladná čísla = zlepšení, záporná = regrese.
baseline_row = summary.loc['v1_baseline']
delta = (summary - baseline_row).drop(index='v1_baseline')
delta.map(lambda x: f'{x:+.3f}')

## 4. Side-by-side v LangSmith UI

Lokální tabulka ukáže means; UI Compare view ti dá **per-example diff** — vidíš input, oba výstupy a oba evaluator verdicts vedle sebe.

**Postup:**
1. Otevři **Datasets & experiments** → `kosik-eval-golden` → tab **Experiments**.
2. Vyfiltruj podle `metadata.ab_group is "<ten ab_group výše>"` (nebo prostě najdi experimenty začínající `ab-<ab_group>-`).
3. Označ checkboxy variant, které chceš porovnat.
4. Klik **Compare** vlevo nahoře.
5. V Compare view klikni na řádek = jeden example → vidíš tool calls, finální odpovědi a evaluator skóre obou variant.

**Co hledat:**
- Zlepšila se cílová metrika? (V2 → `cites_product_id`, V3 → `allergen_flagged_explicitly`)
- Nezhoršila se jiná metrika? (regression risk — častý problém)
- Jsou rozdíly **konzistentní** napříč příklady, nebo je to šum?

In [7]:
# Convenience link na compare view tvojí AB skupiny.
import os
from kosik_workshop.evals.dataset import DATASET_NAME

base = os.getenv('LANGSMITH_ENDPOINT', 'https://eu.smith.langchain.com').rstrip('/').replace('api.', '')
print(f'1) Otevři: {base}/datasets')
print(f'2) Najdi dataset: {DATASET_NAME}')
print(f'3) Tab Experiments → filter ab_group = "{ab_group}"')

1) Otevři: https://eu.smith.langchain.com/datasets
2) Najdi dataset: kosik-eval-golden
3) Tab Experiments → filter ab_group = "nb-20260426-203133"


## 5. Co dál

- Pokud V2 nebo V3 vyhrály bez regressí — promotni je na nový baseline (`scripts/push_prompt.py` + `scripts/promote_prompt.py`).
- Pokud výsledky jsou mixed (zlepšení v jedné, regrese v jiné) — tradeoff k diskusi se stakeholders.
- Pokud rozdíl je v rozsahu šumu (~0.05) → potřebuješ větší dataset, nebo není change worth shippnout.
- Vlastní variantu? Přidej do `src/kosik_workshop/prompts/variants.py` a pusť tenhle notebook znovu.